# Coronagraphic TDI From Simulated Six Links

This notebook is the first test implementation of coronagraphic TDI on the six-link outputs from the LISA link simulation.

The paper in the repository is `2406.00190.txt` / `2406.00190.clean.txt`. The coronagraphic construction is natively frequency-domain, `kappa_tilde(f, beta, lambda) = A(f, beta, lambda) . (alpha_tilde, beta_tilde, gamma_tilde)`, so this notebook computes it in frequency domain and transforms the result back to a complex time-domain `kappa(t)`.

For this first test we use a static LISA geometry frozen at the signal time from the realistic orbit file. Later we can extend this to time-evolving geometry or a windowed/time-frequency implementation.

In [1]:
from pathlib import Path
import sys

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "lisa_link_sim":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from coronagraphic_tdi import LISAGeometry, compute_sagnac_and_coronagraphic_from_links, coronagraphic_A_vector

BASE_DIR = PROJECT_ROOT / "lisa_link_sim"
INPUT_DIR = BASE_DIR / "inputs"
OUTPUT_DIR = BASE_DIR / "outputs"
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

NOISY_LINKS_CSV = OUTPUT_DIR / "six_links_noisy.csv"
GW_LINKS_CSV = OUTPUT_DIR / "six_links_gw_only.csv"
ORBITS_FILE = INPUT_DIR / "orbits.h5"

C_SI = 299792458.0
LINK_LABELS = ["12", "23", "31", "13", "32", "21"]

PROJECT_ROOT


PosixPath('/Users/shubhanshutiwari/IdeaProjects/LISA-TF')

## Load Six-Link Data

Set `LINK_SOURCE` to `gw_only_hz` for a clean null test, or `noisy` to use the realistic `GW + instrument noise` six links.

In [2]:
LINK_SOURCE = "noisy"  # options: "gw_only_hz", "noisy"
CHUNK_HALF_WIDTH_SECONDS = 12_000.0
SIGNAL_THRESHOLD_FRACTION = 1e-3

if LINK_SOURCE == "noisy":
    links_df = pd.read_csv(NOISY_LINKS_CSV)
    link_columns = {lab: f"noisy_{lab}" for lab in LINK_LABELS}
elif LINK_SOURCE == "gw_only_hz":
    links_df = pd.read_csv(GW_LINKS_CSV)
    links_df = links_df[links_df["is_science"].astype(bool)].reset_index(drop=True)
    link_columns = {lab: f"gw_{lab}_hz" for lab in LINK_LABELS}
else:
    raise ValueError("LINK_SOURCE must be 'gw_only_hz' or 'noisy'")

# Use GW-only links to locate the signal even when the analyzed data include noise.
gw_ref = pd.read_csv(GW_LINKS_CSV)
gw_ref = gw_ref[gw_ref["is_science"].astype(bool)].reset_index(drop=True)
gw_t_lisa = gw_ref["t_lisa"].to_numpy(dtype=float)
gw_matrix = np.column_stack([gw_ref[f"gw_{lab}_hz"].to_numpy(dtype=float) for lab in LINK_LABELS])
gw_power = np.sum(gw_matrix**2, axis=1)
signal_idx_ref = int(np.argmax(gw_power))
signal_t_lisa = float(gw_t_lisa[signal_idx_ref])

signal_threshold = SIGNAL_THRESHOLD_FRACTION * float(np.max(gw_power))
signal_window_idx = np.flatnonzero(gw_power >= signal_threshold)
signal_window_lisa = (
    float(gw_t_lisa[signal_window_idx[0]]),
    float(gw_t_lisa[signal_window_idx[-1]]),
)

t_lisa_all = links_df["t_lisa"].to_numpy(dtype=float)
if "t_waveform" in links_df.columns:
    t_waveform_all = links_df["t_waveform"].to_numpy(dtype=float)
else:
    t_waveform_all = t_lisa_all - t_lisa_all[0]
dt = float(np.median(np.diff(t_waveform_all)))

chunk_mask = np.abs(t_lisa_all - signal_t_lisa) <= CHUNK_HALF_WIDTH_SECONDS
links_chunk = links_df.loc[chunk_mask].reset_index(drop=True)
t_lisa = links_chunk["t_lisa"].to_numpy(dtype=float)
t_rel = t_lisa - t_lisa[0]

link_data = {lab: links_chunk[link_columns[lab]].to_numpy(dtype=float) for lab in LINK_LABELS}

signal_peak_rel = signal_t_lisa - t_lisa[0]
signal_window_rel = (
    max(signal_window_lisa[0], t_lisa[0]) - t_lisa[0],
    min(signal_window_lisa[1], t_lisa[-1]) - t_lisa[0],
)

print(f"source       = {LINK_SOURCE}")
print(f"samples      = {len(links_chunk)}")
print(f"dt           = {dt} s")
print(f"chunk span   = {t_rel[-1] - t_rel[0]:.1f} s")
print(f"signal t_lisa= {signal_t_lisa}")
print(f"signal window= {signal_window_lisa}")
print(f"link max abs = {np.max(np.abs(np.column_stack(list(link_data.values())))):.3e} Hz")


source       = noisy
samples      = 240001
dt           = 0.09999999999854481 s
chunk span   = 24000.0 s
signal t_lisa= 61421303.327664
signal window= (61417651.327664055, 61421532.527664)
link max abs = 3.017e+02 Hz


## Freeze LISA Geometry At Signal Time

The older implementation assumes static geometry. Here we build `LISAGeometry` from `tcb/x` in the realistic orbit HDF5 file at the orbit sample nearest the signal time. Positions are stored in meters and converted to light-seconds.

In [3]:
with h5py.File(ORBITS_FILE, "r") as f:
    orbit_t0 = float(f.attrs["t0"])
    orbit_dt = float(f.attrs["dt"])
    orbit_size = int(f.attrs["size"])
    orbit_index = int(np.clip(round((signal_t_lisa - orbit_t0) / orbit_dt), 0, orbit_size - 1))
    x_m = np.asarray(f["tcb/x"][orbit_index], dtype=float)
    ltt = np.asarray(f["tcb/ltt"][orbit_index], dtype=float)

geometry = LISAGeometry.from_positions(x_m[0], x_m[1], x_m[2], c_m_per_s=C_SI)

print(f"orbit index = {orbit_index}")
print(f"orbit time  = {orbit_t0 + orbit_index * orbit_dt}")
print(f"orbit ltt   = {ltt}")
print("static geometry L [s]:")
for lab in LINK_LABELS:
    print(f"  L_{lab} = {geometry.L[lab]:.6f}")


orbit index = 417
orbit time  = 61421439.327664
orbit ltt   = [8.20936944 8.35588986 8.25496895 8.25332631 8.35666564 8.21022441]
static geometry L [s]:
  L_12 = 8.209797
  L_23 = 8.356278
  L_31 = 8.254147
  L_13 = 8.254147
  L_32 = 8.356278
  L_21 = 8.209797


## Two Sky Directions

The simulation source used `RA = pi/4`, `DEC = 0.3`. In the older coronagraphic implementation the paper variables are named `(beta, lambda)`, with `beta` acting like latitude/declination and `lambda` like longitude/right ascension. The first direction is therefore the signal direction; the second is intentionally far away.

In [4]:
sky_positions = {
    # Injected waveform sky used by LisaLinkSimulationConfig: dec=0.3, ra=pi/4.
    "signal_direction": {"beta": 0.3, "lambda": np.pi / 4.0},
    "sky_offset_1": {"beta": -0.9, "lambda": np.pi / 4.0 + 2.0},
    "sky_offset_2": {"beta": 0.9, "lambda": np.pi / 4.0 - 2.2},
    "sky_offset_3": {"beta": -0.2, "lambda": np.pi / 4.0 + np.pi},
    "sky_offset_4": {"beta": 0.0, "lambda": np.pi / 4.0 + 1.2},
}
sky_positions


{'signal_direction': {'beta': 0.3, 'lambda': 0.7853981633974483},
 'sky_offset_1': {'beta': -0.9, 'lambda': 2.7853981633974483},
 'sky_offset_2': {'beta': 0.9, 'lambda': -1.414601836602552},
 'sky_offset_3': {'beta': -0.2, 'lambda': 3.9269908169872414},
 'sky_offset_4': {'beta': 0.0, 'lambda': 1.9853981633974482}}

## Cross-Check Injected Sky Direction

The six-link generation notebook uses `RA = pi/4` and `DEC = 0.3` in `ReadStrain`. The coronagraphic code uses the paper notation `(beta, lambda)`, where this implementation treats `beta` as declination/latitude and `lambda` as right ascension/longitude.


In [5]:
INJECTED_RA = np.pi / 4.0
INJECTED_DEC = 0.3

signal_sky = sky_positions["signal_direction"]
sky_check_rows = [
    {
        "quantity": "right_ascension_vs_lambda",
        "generated_value": INJECTED_RA,
        "tested_value": signal_sky["lambda"],
        "abs_difference": abs(INJECTED_RA - signal_sky["lambda"]),
        "matches": bool(np.isclose(INJECTED_RA, signal_sky["lambda"])),
    },
    {
        "quantity": "declination_vs_beta",
        "generated_value": INJECTED_DEC,
        "tested_value": signal_sky["beta"],
        "abs_difference": abs(INJECTED_DEC - signal_sky["beta"]),
        "matches": bool(np.isclose(INJECTED_DEC, signal_sky["beta"])),
    },
]

try:
    from scripts.lisa_link_simulation import LisaLinkSimulationConfig

    default_config = LisaLinkSimulationConfig()
    sky_check_rows.extend([
        {
            "quantity": "script_default_ra_vs_signal_lambda",
            "generated_value": default_config.ra,
            "tested_value": signal_sky["lambda"],
            "abs_difference": abs(default_config.ra - signal_sky["lambda"]),
            "matches": bool(np.isclose(default_config.ra, signal_sky["lambda"])),
        },
        {
            "quantity": "script_default_dec_vs_signal_beta",
            "generated_value": default_config.dec,
            "tested_value": signal_sky["beta"],
            "abs_difference": abs(default_config.dec - signal_sky["beta"]),
            "matches": bool(np.isclose(default_config.dec, signal_sky["beta"])),
        },
    ])
except Exception as exc:
    sky_check_rows.append({
        "quantity": "script_default_config_import",
        "generated_value": np.nan,
        "tested_value": np.nan,
        "abs_difference": np.nan,
        "matches": False,
        "note": str(exc),
    })

sky_check_df = pd.DataFrame(sky_check_rows)
SKY_CHECK_CSV = OUTPUT_DIR / f"coronagraphic_tdi_sky_direction_check_{LINK_SOURCE}.csv"
sky_check_df.to_csv(SKY_CHECK_CSV, index=False)

if not sky_check_df["matches"].all():
    raise ValueError("Injected sky direction does not match signal_direction. See sky_check_df.")

print(f"saved {SKY_CHECK_CSV}")
sky_check_df


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/coronagraphic_tdi_sky_direction_check_noisy.csv


,quantity,generated_value,tested_value,abs_difference,matches
0,right_ascension_vs_lambda,0.785398,0.785398,0.0,True
1,declination_vs_beta,0.300000,0.300000,0.0,True
2,script_default_ra_vs_signal_lambda,0.785398,0.785398,0.0,True
3,script_default_dec_vs_signal_beta,0.300000,0.300000,0.0,True


## Compute Time-Domain Coronagraphic TDI

This calls the old frequency-domain construction, then returns complex time-domain `alpha(t)`, `beta(t)`, `gamma(t)`, and `kappa(t)` through the inverse transform.

In [6]:
results = {}
for name, pars in sky_positions.items():
    out = compute_sagnac_and_coronagraphic_from_links(
        link_data,
        dt=dt,
        geometry=geometry,
        beta=pars["beta"],
        lamb=pars["lambda"],
        output_domain="both",
    )
    results[name] = out

summary = []
for name, out in results.items():
    kappa_t = out["time"]["kappa"]
    kappa_f = out["frequency"]["kappa"]
    summary.append({
        "sky": name,
        "beta": sky_positions[name]["beta"],
        "lambda": sky_positions[name]["lambda"],
        "time_abs_max": np.max(np.abs(kappa_t)),
        "time_abs_rms": np.sqrt(np.mean(np.abs(kappa_t) ** 2)),
        "freq_abs_max": np.max(np.abs(kappa_f)),
    })

summary_df = pd.DataFrame(summary)
summary_df


,sky,beta,lambda,time_abs_max,time_abs_rms,freq_abs_max
0,signal_direction,0.3,0.785398,307.313966,46.060873,362272.212434
1,sky_offset_1,-0.9,2.785398,36.767260,3.641880,35570.491923
2,sky_offset_2,0.9,-1.414602,394.934319,43.499183,399425.817943
3,sky_offset_3,-0.2,3.926991,340.744114,39.976665,311365.495960
4,sky_offset_4,0.0,1.985398,330.207119,50.501579,388140.504326


## Diagnostic: GW-Only Null Versus Noise Transfer

The coronagraphic null applies to the GW signal. Raw noisy `kappa` also contains sky-dependent noise shaping, so this cell computes a GW-only reference on the same chunk and the norm of the coronagraphic coefficient vector `A(f, beta, lambda)`.

In [7]:
# Build a GW-only reference on the same time chunk, even when LINK_SOURCE == "noisy".
gw_ref_chunk = gw_ref[(gw_ref["t_lisa"] >= t_lisa[0]) & (gw_ref["t_lisa"] <= t_lisa[-1])].reset_index(drop=True)
gw_link_data = {lab: gw_ref_chunk[f"gw_{lab}_hz"].to_numpy(dtype=float) for lab in LINK_LABELS}

if len(gw_ref_chunk) != len(t_lisa):
    raise ValueError("GW-only reference chunk does not match selected chunk length.")

gw_only_results = {}
for name, pars in sky_positions.items():
    gw_only_results[name] = compute_sagnac_and_coronagraphic_from_links(
        gw_link_data,
        dt=dt,
        geometry=geometry,
        beta=pars["beta"],
        lamb=pars["lambda"],
        output_domain="both",
    )

rows = []
for name, pars in sky_positions.items():
    k_selected = results[name]["time"]["kappa"]
    k_gw = gw_only_results[name]["time"]["kappa"]
    freqs = results[name]["freqs_hz"]
    band = (np.abs(freqs) >= 1e-5) & (np.abs(freqs) <= 1e-2)
    avec = coronagraphic_A_vector(freqs, pars["beta"], pars["lambda"], geometry)
    a_norm = np.sqrt(np.sum(np.abs(avec) ** 2, axis=0))
    rows.append({
        "sky": name,
        "selected_source": LINK_SOURCE,
        "selected_kappa_rms": np.sqrt(np.mean(np.abs(k_selected) ** 2)),
        "gw_only_kappa_rms": np.sqrt(np.mean(np.abs(k_gw) ** 2)),
        "gw_only_kappa_max": np.max(np.abs(k_gw)),
        "A_norm_median_1e-5_1e-2": np.median(a_norm[band]),
        "A_norm_mean_1e-5_1e-2": np.mean(a_norm[band]),
    })

diagnostic_df = pd.DataFrame(rows)
diagnostic_df


,sky,selected_source,selected_kappa_rms,gw_only_kappa_rms,gw_only_kappa_max,A_norm_median_1e-5_1e-2,A_norm_mean_1e-5_1e-2
0,signal_direction,noisy,46.060873,8.489828e-08,0.000007,0.014642,0.044093
1,sky_offset_1,noisy,3.641880,7.318390e-07,0.000047,0.000679,0.002044
2,sky_offset_2,noisy,43.499183,4.015221e-06,0.000211,0.008884,0.026729
3,sky_offset_3,noisy,39.976665,1.110449e-05,0.000443,0.015905,0.047916
4,sky_offset_4,noisy,50.501579,8.250831e-06,0.000341,0.009145,0.027510


## Noise-Normalized Off-Source Statistics

Estimate the kappa noise PSD separately for each sky direction from off-source data in the same noisy realization, excluding the GW signal interval plus a guard region. This normalizes out the sky-dependent coronagraphic transfer function before comparing directions.


In [8]:
LOW_FREQ_CUT = 1e-5
HIGH_FREQ_CUT = 1e-2
N_FREQUENCY_BINS = 3
FREQUENCY_BIN_EDGES = np.geomspace(LOW_FREQ_CUT, HIGH_FREQ_CUT, N_FREQUENCY_BINS + 1)
OFFSOURCE_GUARD_SECONDS = 500.0
WELCH_NPERSEG = 16_384


def contiguous_true_ranges(mask):
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return []
    breaks = np.flatnonzero(np.diff(idx) > 1) + 1
    chunks = np.split(idx, breaks)
    return [(int(chunk[0]), int(chunk[-1]) + 1) for chunk in chunks if len(chunk)]


def estimate_kappa_psd_offsource(kappa_t, dt, offsource_mask, nperseg=WELCH_NPERSEG):
    """Welch estimate for complex kappa using the same +FT convention as coronagraphic_tdi."""
    nperseg = int(min(nperseg, len(kappa_t)))
    if nperseg < 256:
        raise ValueError("Need at least 256 samples for the off-source PSD estimate.")

    ranges = contiguous_true_ranges(offsource_mask)
    longest = max((hi - lo for lo, hi in ranges), default=0)
    while nperseg > longest and nperseg > 256:
        nperseg //= 2

    if nperseg > longest:
        raise ValueError("No off-source segment is long enough for a PSD estimate.")

    step = max(nperseg // 2, 1)
    window = np.hanning(nperseg)
    window_power = np.sum(window**2)
    psd_segments = []

    for lo, hi in ranges:
        if hi - lo < nperseg:
            continue
        for start in range(lo, hi - nperseg + 1, step):
            segment = np.asarray(kappa_t[start:start + nperseg], dtype=complex)
            segment = segment - np.mean(segment)
            # The existing coronagraphic module uses ifft(x) * N for its positive-sign FT.
            spectrum = np.fft.ifft(segment * window) * nperseg
            psd_segments.append(dt * np.abs(spectrum) ** 2 / window_power)

    if not psd_segments:
        raise ValueError("Welch PSD produced no off-source segments.")

    freqs = np.fft.fftfreq(nperseg, d=dt)
    psd = np.mean(np.vstack(psd_segments), axis=0)
    return freqs, psd, len(psd_segments), nperseg


signal_mask_with_guard = (
    (t_lisa >= signal_window_lisa[0] - OFFSOURCE_GUARD_SECONDS)
    & (t_lisa <= signal_window_lisa[1] + OFFSOURCE_GUARD_SECONDS)
)
offsource_mask = ~signal_mask_with_guard

noise_normalized = {}
stat_rows = []
frequency_bin_rows = []
psd_table = None

for name, out in results.items():
    kappa_t = out["time"]["kappa"]
    full_freqs = out["freqs_hz"]
    kappa_f = out["frequency"]["kappa"]

    psd_freqs, psd, n_segments, nperseg_used = estimate_kappa_psd_offsource(
        kappa_t, dt, offsource_mask
    )

    psd_order = np.argsort(psd_freqs)
    psd_freqs_sorted = psd_freqs[psd_order]
    psd_sorted = psd[psd_order]
    s_interp = np.interp(full_freqs, psd_freqs_sorted, psd_sorted, left=np.nan, right=np.nan)

    pos_band = (
        (full_freqs >= LOW_FREQ_CUT)
        & (full_freqs <= HIGH_FREQ_CUT)
        & np.isfinite(s_interp)
        & (s_interp > 0)
    )
    normalized_power = np.abs(kappa_f[pos_band]) ** 2 / s_interp[pos_band]

    signal_core_mask = (t_lisa >= signal_window_lisa[0]) & (t_lisa <= signal_window_lisa[1])
    offsource_abs_power = np.abs(kappa_t[offsource_mask]) ** 2
    signal_abs_power = np.abs(kappa_t[signal_core_mask]) ** 2
    offsource_power_mean = float(np.mean(offsource_abs_power))
    offsource_power_std = float(np.std(offsource_abs_power, ddof=1))
    signal_power_mean = float(np.mean(signal_abs_power))
    signal_power_stderr = offsource_power_std / np.sqrt(np.sum(signal_core_mask))

    whitened_abs = np.full_like(full_freqs, np.nan, dtype=float)
    valid = np.isfinite(s_interp) & (s_interp > 0)
    whitened_abs[valid] = np.abs(kappa_f[valid]) / np.sqrt(s_interp[valid])

    noise_normalized[name] = {
        "psd_freqs_hz": psd_freqs,
        "psd": psd,
        "s_interp": s_interp,
        "whitened_abs": whitened_abs,
        "band_mask": pos_band,
        "normalized_power": normalized_power,
        "welch_segments": n_segments,
        "welch_nperseg": nperseg_used,
    }

    for bin_index, (bin_low, bin_high) in enumerate(zip(FREQUENCY_BIN_EDGES[:-1], FREQUENCY_BIN_EDGES[1:]), start=1):
        bin_mask = (
            (full_freqs >= bin_low)
            & (full_freqs < bin_high if bin_index < N_FREQUENCY_BINS else full_freqs <= bin_high)
            & np.isfinite(s_interp)
            & (s_interp > 0)
        )
        bin_normalized_power = np.abs(kappa_f[bin_mask]) ** 2 / s_interp[bin_mask]
        frequency_bin_rows.append({
            "sky": name,
            "source": LINK_SOURCE,
            "frequency_bin": bin_index,
            "bin_low_hz": float(bin_low),
            "bin_high_hz": float(bin_high),
            "freq_bins_used": int(np.sum(bin_mask)),
            "raw_band_power_sum": float(np.sum(np.abs(kappa_f[bin_mask]) ** 2)),
            "noise_norm_power_sum": float(np.sum(bin_normalized_power)),
            "noise_norm_power_mean": float(np.mean(bin_normalized_power)),
            "noise_norm_power_median": float(np.median(bin_normalized_power)),
            "whitened_abs_max": float(np.nanmax(whitened_abs[bin_mask])),
            "whitened_abs_rms": float(np.sqrt(np.nanmean(whitened_abs[bin_mask] ** 2))),
        })

    stat_rows.append({
        "sky": name,
        "source": LINK_SOURCE,
        "low_freq_cut_hz": LOW_FREQ_CUT,
        "high_freq_cut_hz": HIGH_FREQ_CUT,
        "offsource_guard_s": OFFSOURCE_GUARD_SECONDS,
        "offsource_duration_s": float(np.sum(offsource_mask) * dt),
        "welch_segments": n_segments,
        "welch_nperseg": nperseg_used,
        "freq_bins_used": int(np.sum(pos_band)),
        "raw_band_power_sum": float(np.sum(np.abs(kappa_f[pos_band]) ** 2)),
        "time_signal_power_mean": signal_power_mean,
        "time_offsource_power_mean": offsource_power_mean,
        "time_signal_power_over_offsource": signal_power_mean / offsource_power_mean,
        "time_signal_abs_rms_over_offsource": np.sqrt(signal_power_mean / offsource_power_mean),
        "time_signal_power_excess_z_naive": (signal_power_mean - offsource_power_mean) / signal_power_stderr,
        "noise_norm_power_sum": float(np.sum(normalized_power)),
        "noise_norm_power_mean": float(np.mean(normalized_power)),
        "noise_norm_power_median": float(np.median(normalized_power)),
        "whitened_abs_max": float(np.nanmax(whitened_abs[pos_band])),
        "whitened_abs_rms": float(np.sqrt(np.nanmean(whitened_abs[pos_band] ** 2))),
    })

    psd_positive = psd_freqs > 0
    this_psd = pd.DataFrame({
        "frequency_hz": psd_freqs[psd_positive],
        f"S_kappa_{name}": psd[psd_positive],
    })
    if psd_table is None:
        psd_table = this_psd.sort_values("frequency_hz").reset_index(drop=True)
    else:
        psd_table = psd_table.merge(
            this_psd.sort_values("frequency_hz"), on="frequency_hz", how="outer"
        )

noise_norm_df = pd.DataFrame(stat_rows)
frequency_bin_df = pd.DataFrame(frequency_bin_rows)
NOISE_NORM_STATS_CSV = OUTPUT_DIR / f"coronagraphic_tdi_noise_normalized_stats_{LINK_SOURCE}.csv"
NOISE_NORM_BIN_STATS_CSV = OUTPUT_DIR / f"coronagraphic_tdi_noise_normalized_stats_3bins_{LINK_SOURCE}.csv"
NOISE_PSD_CSV = OUTPUT_DIR / f"coronagraphic_tdi_offsource_psd_{LINK_SOURCE}.csv"
noise_norm_df.to_csv(NOISE_NORM_STATS_CSV, index=False)
frequency_bin_df.to_csv(NOISE_NORM_BIN_STATS_CSV, index=False)
psd_table.sort_values("frequency_hz").to_csv(NOISE_PSD_CSV, index=False)

print(f"saved {NOISE_NORM_STATS_CSV}")
print(f"saved {NOISE_NORM_BIN_STATS_CSV}")
print(f"saved {NOISE_PSD_CSV}")
frequency_bin_df.sort_values(["frequency_bin", "noise_norm_power_mean"], ascending=[True, False])


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/coronagraphic_tdi_noise_normalized_stats_noisy.csv
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/coronagraphic_tdi_noise_normalized_stats_3bins_noisy.csv
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/coronagraphic_tdi_offsource_psd_noisy.csv


,sky,source,frequency_bin,bin_low_hz,bin_high_hz,freq_bins_used,raw_band_power_sum,noise_norm_power_sum,noise_norm_power_mean,noise_norm_power_median,whitened_abs_max,whitened_abs_rms
9,sky_offset_3,noisy,1,0.00001,0.0001,2,3.517322e-15,1.550992e-13,7.754959e-14,7.754959e-14,3.931950e-07,2.784773e-07
0,signal_direction,noisy,1,0.00001,0.0001,2,2.981416e-15,6.756847e-14,3.378423e-14,3.378423e-14,2.595227e-07,1.838049e-07
6,sky_offset_2,noisy,1,0.00001,0.0001,2,1.098872e-15,3.713445e-14,1.856723e-14,1.856723e-14,1.923941e-07,1.362616e-07
12,sky_offset_4,noisy,1,0.00001,0.0001,2,1.164618e-15,1.997099e-14,9.985496e-15,9.985496e-15,1.410922e-07,9.992746e-08
3,sky_offset_1,noisy,1,0.00001,0.0001,2,6.408774e-18,1.926026e-14,9.630129e-15,9.630129e-15,1.385589e-07,9.813322e-08
10,sky_offset_3,noisy,2,0.00010,0.0010,22,4.566440e-06,1.546552e-03,7.029782e-05,1.876678e-06,2.491819e-02,8.384379e-03
1,signal_direction,noisy,2,0.00010,0.0010,22,3.808924e-06,6.624886e-04,3.011312e-05,8.125298e-07,1.629077e-02,5.487542e-03
7,sky_offset_2,noisy,2,0.00010,0.0010,22,1.398779e-06,3.625891e-04,1.648132e-05,4.458178e-07,1.204897e-02,4.059719e-03
13,sky_offset_4,noisy,2,0.00010,0.0010,22,1.509996e-06,1.988594e-04,9.039063e-06,2.415199e-07,8.934446e-03,3.006503e-03
4,sky_offset_1,noisy,2,0.00010,0.0010,22,8.387971e-09,1.936747e-04,8.803394e-06,2.337638e-07,8.822890e-03,2.967051e-03


## Noise-Normalized Plots


In [9]:
plt.figure(figsize=(13, 5))
for name, out in results.items():
    off_rms = float(np.sqrt(noise_norm_df.loc[noise_norm_df["sky"] == name, "time_offsource_power_mean"].iloc[0]))
    plt.plot(t_rel, np.abs(out["time"]["kappa"]) / off_rms, lw=0.9, label=f"off-source normalized |kappa| {name}")
plt.axvspan(signal_window_rel[0], signal_window_rel[1], color="gold", alpha=0.20, label="GW signal interval")
plt.axvline(signal_peak_rel, color="k", ls="--", alpha=0.6, label="GW signal peak")
plt.xlabel("time in chunk [s]")
plt.ylabel(r"$|\kappa(t)| / \mathrm{rms}_{\mathrm{off}}$")
plt.title(f"Time-domain off-source normalized coronagraphic TDI from {LINK_SOURCE} links")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
PLOT_NOISE_NORM_TIME = PLOT_DIR / f"08_coronagraphic_tdi_noise_normalized_time_{LINK_SOURCE}.png"
plt.savefig(PLOT_NOISE_NORM_TIME, dpi=150)
plt.show()
print(f"saved {PLOT_NOISE_NORM_TIME}")

plt.figure(figsize=(13, 5))
for name, nn in noise_normalized.items():
    freqs = results[name]["freqs_hz"]
    plot_mask = (freqs >= LOW_FREQ_CUT) & (freqs <= HIGH_FREQ_CUT) & np.isfinite(nn["whitened_abs"])
    plt.loglog(freqs[plot_mask], nn["whitened_abs"][plot_mask], lw=0.9, label=f"whitened |kappa| {name}")
plt.xlabel("frequency [Hz]")
plt.ylabel(r"$|\tilde\kappa| / \sqrt{S_\kappa}$")
plt.title(f"Off-source noise-normalized coronagraphic statistic from {LINK_SOURCE} links")
plt.grid(alpha=0.3, which="both")
plt.legend()
plt.tight_layout()
PLOT_NOISE_NORM_FREQ = PLOT_DIR / f"08_coronagraphic_tdi_noise_normalized_frequency_{LINK_SOURCE}.png"
plt.savefig(PLOT_NOISE_NORM_FREQ, dpi=150)
plt.show()
print(f"saved {PLOT_NOISE_NORM_FREQ}")

plt.figure(figsize=(7, 4))
bar_colors = plt.cm.tab10(np.linspace(0, 1, len(noise_norm_df)))
plt.bar(noise_norm_df["sky"], noise_norm_df["noise_norm_power_mean"], color=bar_colors)
plt.ylabel(r"mean $|\tilde\kappa|^2 / S_\kappa$")
plt.title(f"Band-averaged noise-normalized power, {LOW_FREQ_CUT:g}-{HIGH_FREQ_CUT:g} Hz")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
PLOT_NOISE_NORM_STAT = PLOT_DIR / f"08_coronagraphic_tdi_noise_normalized_stat_{LINK_SOURCE}.png"
plt.savefig(PLOT_NOISE_NORM_STAT, dpi=150)
plt.show()
print(f"saved {PLOT_NOISE_NORM_STAT}")


fig, axes = plt.subplots(1, N_FREQUENCY_BINS, figsize=(15, 4), sharey=True)
if N_FREQUENCY_BINS == 1:
    axes = [axes]
for ax, bin_index in zip(axes, range(1, N_FREQUENCY_BINS + 1)):
    bin_df = frequency_bin_df[frequency_bin_df["frequency_bin"] == bin_index].sort_values(
        "noise_norm_power_mean", ascending=False
    )
    colors = plt.cm.tab10(np.linspace(0, 1, len(bin_df)))
    ax.bar(bin_df["sky"], bin_df["noise_norm_power_mean"], color=colors)
    low = bin_df["bin_low_hz"].iloc[0]
    high = bin_df["bin_high_hz"].iloc[0]
    ax.set_title(f"Bin {bin_index}: {low:.1e}-{high:.1e} Hz")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel(r"mean $|\tilde\kappa|^2 / S_\kappa$")
fig.suptitle(f"Three-bin off-source normalized coronagraphic statistic from {LINK_SOURCE} links")
fig.tight_layout()
PLOT_NOISE_NORM_3BINS = PLOT_DIR / f"09_coronagraphic_tdi_noise_normalized_3bins_{LINK_SOURCE}.png"
fig.savefig(PLOT_NOISE_NORM_3BINS, dpi=150)
plt.show()
print(f"saved {PLOT_NOISE_NORM_3BINS}")


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3967603640.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3967603640.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3967603640.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/08_coronagraphic_tdi_noise_normalized_time_noisy.png
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/08_coronagraphic_tdi_noise_normalized_frequency_noisy.png
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/08_coronagraphic_tdi_noise_normalized_stat_noisy.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/09_coronagraphic_tdi_noise_normalized_3bins_noisy.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3967603640.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## WDM Time-Frequency Noise-Normalized Statistic

Use `wdm-wavelet` on a downsampled coronagraphic kappa time series. The statistic is normalized per WDM frequency layer using off-source time pixels, then evaluated inside the GW signal time window and within the analysis frequency band.


In [10]:
from scipy.signal import decimate
from wdm_wavelet.wdm import WDM

WDM_TARGET_DT = 1.0
WDM_M = 1024
WDM_K = 32
WDM_BETA_ORDER = 6
WDM_PRECISION = 6
WDM_BACKEND = "numba"
WDM_LOW_FREQ_CUT = 1e-5
WDM_HIGH_FREQ_CUT = 1e-2
WDM_MIN_NOISE_FLOOR = 1e-300


def downsample_complex_for_wdm(x, dt, target_dt):
    factor = int(round(target_dt / dt))
    if not np.isclose(factor * dt, target_dt):
        raise ValueError("WDM_TARGET_DT must be an integer multiple of the native dt.")
    if factor <= 1:
        return np.asarray(x, dtype=complex), dt
    real = decimate(np.real(x), factor, ftype="fir", zero_phase=True)
    imag = decimate(np.imag(x), factor, ftype="fir", zero_phase=True)
    return real + 1j * imag, factor * dt


def wdm_complex_power(wdm, x_complex, sample_rate, t0):
    tf_real = wdm.t2w(np.real(x_complex), sample_rate=sample_rate, t0=t0)
    tf_imag = wdm.t2w(np.imag(x_complex), sample_rate=sample_rate, t0=t0)
    power = np.abs(tf_real.data) ** 2 + np.abs(tf_imag.data) ** 2
    freqs = np.arange(tf_real.n_freq, dtype=float) * tf_real.df
    times = tf_real.t0 + (np.arange(tf_real.n_time, dtype=float) + 0.5) * tf_real.dt
    return power, times, freqs, tf_real


wdm = WDM(M=WDM_M, K=WDM_K, beta_order=WDM_BETA_ORDER, precision=WDM_PRECISION, backend=WDM_BACKEND)
wdm_rows = []
wdm_bin_rows = []
wdm_products = {}

for name, out in results.items():
    kappa_ds, dt_wdm = downsample_complex_for_wdm(out["time"]["kappa"], dt, WDM_TARGET_DT)
    sample_rate_wdm = 1.0 / dt_wdm
    power_tf, wdm_times, wdm_freqs, tf_template = wdm_complex_power(
        wdm, kappa_ds, sample_rate=sample_rate_wdm, t0=t_lisa[0]
    )

    signal_tf_mask = (wdm_times >= signal_window_lisa[0]) & (wdm_times <= signal_window_lisa[1])
    offsource_tf_mask = (
        (wdm_times < signal_window_lisa[0] - OFFSOURCE_GUARD_SECONDS)
        | (wdm_times > signal_window_lisa[1] + OFFSOURCE_GUARD_SECONDS)
    )
    freq_tf_mask = (wdm_freqs >= WDM_LOW_FREQ_CUT) & (wdm_freqs <= WDM_HIGH_FREQ_CUT)

    noise_floor = np.nanmedian(power_tf[:, offsource_tf_mask], axis=1)
    noise_floor = np.where(np.isfinite(noise_floor) & (noise_floor > 0), noise_floor, WDM_MIN_NOISE_FLOOR)
    normalized_tf_power = power_tf / noise_floor[:, None]

    signal_band = normalized_tf_power[np.ix_(freq_tf_mask, signal_tf_mask)]
    offsource_band = normalized_tf_power[np.ix_(freq_tf_mask, offsource_tf_mask)]

    wdm_products[name] = {
        "power": power_tf,
        "normalized_power": normalized_tf_power,
        "times_lisa": wdm_times,
        "times_rel": wdm_times - t_lisa[0],
        "freqs_hz": wdm_freqs,
        "noise_floor": noise_floor,
        "signal_tf_mask": signal_tf_mask,
        "offsource_tf_mask": offsource_tf_mask,
        "freq_tf_mask": freq_tf_mask,
        "tf_template": tf_template,
    }

    wdm_rows.append({
        "sky": name,
        "source": LINK_SOURCE,
        "wdm_dt_input_s": dt_wdm,
        "wdm_M": WDM_M,
        "wdm_K": WDM_K,
        "wdm_df_hz": float(tf_template.df),
        "wdm_dt_tf_s": float(tf_template.dt),
        "n_signal_time_pixels": int(np.sum(signal_tf_mask)),
        "n_offsource_time_pixels": int(np.sum(offsource_tf_mask)),
        "n_frequency_pixels": int(np.sum(freq_tf_mask)),
        "signal_tf_pixels": int(signal_band.size),
        "wdm_signal_norm_power_sum": float(np.nansum(signal_band)),
        "wdm_signal_norm_power_mean": float(np.nanmean(signal_band)),
        "wdm_signal_norm_power_median": float(np.nanmedian(signal_band)),
        "wdm_signal_norm_power_max": float(np.nanmax(signal_band)),
        "wdm_offsource_norm_power_mean": float(np.nanmean(offsource_band)),
        "wdm_signal_over_offsource_mean": float(np.nanmean(signal_band) / np.nanmean(offsource_band)),
    })

    for bin_index, (bin_low, bin_high) in enumerate(zip(FREQUENCY_BIN_EDGES[:-1], FREQUENCY_BIN_EDGES[1:]), start=1):
        bin_freq_mask = (
            (wdm_freqs >= bin_low)
            & (wdm_freqs < bin_high if bin_index < N_FREQUENCY_BINS else wdm_freqs <= bin_high)
        )
        if not np.any(bin_freq_mask):
            bin_signal = np.array([np.nan])
            bin_offsource = np.array([np.nan])
        else:
            bin_signal = normalized_tf_power[np.ix_(bin_freq_mask, signal_tf_mask)]
            bin_offsource = normalized_tf_power[np.ix_(bin_freq_mask, offsource_tf_mask)]
        wdm_bin_rows.append({
            "sky": name,
            "source": LINK_SOURCE,
            "frequency_bin": bin_index,
            "bin_low_hz": float(bin_low),
            "bin_high_hz": float(bin_high),
            "wdm_frequency_layers": int(np.sum(bin_freq_mask)),
            "signal_tf_pixels": int(np.size(bin_signal)) if np.any(bin_freq_mask) else 0,
            "wdm_signal_norm_power_sum": float(np.nansum(bin_signal)),
            "wdm_signal_norm_power_mean": float(np.nanmean(bin_signal)),
            "wdm_signal_norm_power_median": float(np.nanmedian(bin_signal)),
            "wdm_signal_norm_power_max": float(np.nanmax(bin_signal)),
            "wdm_offsource_norm_power_mean": float(np.nanmean(bin_offsource)),
            "wdm_signal_over_offsource_mean": float(np.nanmean(bin_signal) / np.nanmean(bin_offsource)),
        })

wdm_stats_df = pd.DataFrame(wdm_rows)
wdm_bin_stats_df = pd.DataFrame(wdm_bin_rows)
WDM_STATS_CSV = OUTPUT_DIR / f"coronagraphic_tdi_wdm_tf_stats_{LINK_SOURCE}.csv"
WDM_BIN_STATS_CSV = OUTPUT_DIR / f"coronagraphic_tdi_wdm_tf_stats_3bins_{LINK_SOURCE}.csv"
wdm_stats_df.to_csv(WDM_STATS_CSV, index=False)
wdm_bin_stats_df.to_csv(WDM_BIN_STATS_CSV, index=False)

print(f"saved {WDM_STATS_CSV}")
print(f"saved {WDM_BIN_STATS_CSV}")
print(f"WDM df={wdm_stats_df['wdm_df_hz'].iloc[0]:.3e} Hz, dt_tf={wdm_stats_df['wdm_dt_tf_s'].iloc[0]:.1f} s")
wdm_stats_df.sort_values("wdm_signal_over_offsource_mean", ascending=False)


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/coronagraphic_tdi_wdm_tf_stats_noisy.csv
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/coronagraphic_tdi_wdm_tf_stats_3bins_noisy.csv
WDM df=4.883e-04 Hz, dt_tf=1000.0 s


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/851079297.py:114: RuntimeWarning: Mean of empty slice
  "wdm_signal_norm_power_mean": float(np.nanmean(bin_signal)),
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/851079297.py:115: RuntimeWarning: All-NaN slice encountered
  "wdm_signal_norm_power_median": float(np.nanmedian(bin_signal)),
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/851079297.py:116: RuntimeWarning: All-NaN slice encountered
  "wdm_signal_norm_power_max": float(np.nanmax(bin_signal)),
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/851079297.py:117: RuntimeWarning: Mean of empty slice
  "wdm_offsource_norm_power_mean": float(np.nanmean(bin_offsource)),
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/851079297.py:118: RuntimeWarning: Mean of empty slice
  "wdm_signal_over_offsource_mean": float(np.nanmean(bin_signal) / np.nanmean(bin_offsource)),


,sky,source,wdm_dt_input_s,wdm_M,wdm_K,wdm_df_hz,wdm_dt_tf_s,n_signal_time_pixels,n_offsource_time_pixels,n_frequency_pixels,signal_tf_pixels,wdm_signal_norm_power_sum,wdm_signal_norm_power_mean,wdm_signal_norm_power_median,wdm_signal_norm_power_max,wdm_offsource_norm_power_mean,wdm_signal_over_offsource_mean
0,signal_direction,noisy,1.0,1024,32,0.000488,1000.041667,4,19,20,80,45.595374,0.569942,0.561978,1.533576,3.609078,0.157919
3,sky_offset_3,noisy,1.0,1024,32,0.000488,1000.041667,4,19,20,80,81.150802,1.014385,0.556728,6.093615,8.793699,0.115354
4,sky_offset_4,noisy,1.0,1024,32,0.000488,1000.041667,4,19,20,80,168.912485,2.111406,0.931372,11.927121,499.072197,0.004231
1,sky_offset_1,noisy,1.0,1024,32,0.000488,1000.041667,4,19,20,80,90.758719,1.134484,0.610384,5.958708,389.954384,0.002909
2,sky_offset_2,noisy,1.0,1024,32,0.000488,1000.041667,4,19,20,80,79.902818,0.998785,0.899392,2.793076,1318.084001,0.000758


## WDM Time-Frequency Plots


In [11]:
plt.figure(figsize=(8, 4))
ranked = wdm_stats_df.sort_values("wdm_signal_over_offsource_mean", ascending=False)
colors = plt.cm.tab10(np.linspace(0, 1, len(ranked)))
plt.bar(ranked["sky"], ranked["wdm_signal_over_offsource_mean"], color=colors)
plt.ylabel("signal-window / off-source normalized TF mean")
plt.title("WDM time-frequency normalized statistic")
plt.grid(axis="y", alpha=0.3)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
PLOT_WDM_STAT = PLOT_DIR / f"10_coronagraphic_tdi_wdm_tf_stat_{LINK_SOURCE}.png"
plt.savefig(PLOT_WDM_STAT, dpi=150)
plt.show()
print(f"saved {PLOT_WDM_STAT}")

fig, axes = plt.subplots(1, N_FREQUENCY_BINS, figsize=(15, 4), sharey=True)
if N_FREQUENCY_BINS == 1:
    axes = [axes]
for ax, bin_index in zip(axes, range(1, N_FREQUENCY_BINS + 1)):
    bin_df = wdm_bin_stats_df[wdm_bin_stats_df["frequency_bin"] == bin_index].sort_values(
        "wdm_signal_over_offsource_mean", ascending=False
    )
    colors = plt.cm.tab10(np.linspace(0, 1, len(bin_df)))
    ax.bar(bin_df["sky"], bin_df["wdm_signal_over_offsource_mean"], color=colors)
    low = bin_df["bin_low_hz"].iloc[0]
    high = bin_df["bin_high_hz"].iloc[0]
    layers = int(bin_df["wdm_frequency_layers"].max())
    ax.set_title(f"Bin {bin_index}: {low:.1e}-{high:.1e} Hz\n{layers} WDM layers")
    ax.tick_params(axis="x", rotation=45)
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("signal/off-source TF mean")
fig.suptitle("WDM time-frequency statistic by frequency bin")
fig.tight_layout()
PLOT_WDM_3BINS = PLOT_DIR / f"10_coronagraphic_tdi_wdm_tf_3bins_{LINK_SOURCE}.png"
fig.savefig(PLOT_WDM_3BINS, dpi=150)
plt.show()
print(f"saved {PLOT_WDM_3BINS}")

# Show TF maps for the injected direction and the best-ranked direction.
best_sky = ranked["sky"].iloc[0]
plot_skies = list(dict.fromkeys(["signal_direction", best_sky]))
fig, axes = plt.subplots(1, len(plot_skies), figsize=(7 * len(plot_skies), 4), sharey=True)
if len(plot_skies) == 1:
    axes = [axes]
for ax, name in zip(axes, plot_skies):
    prod = wdm_products[name]
    freqs = prod["freqs_hz"]
    times_rel_wdm = prod["times_rel"]
    fmask = (freqs >= 0.0) & (freqs <= WDM_HIGH_FREQ_CUT)
    image = np.log10(prod["normalized_power"][fmask] + 1e-30)
    extent = [
        times_rel_wdm[0] - 0.5 * prod["tf_template"].dt,
        times_rel_wdm[-1] + 0.5 * prod["tf_template"].dt,
        freqs[fmask][0],
        freqs[fmask][-1],
    ]
    im = ax.imshow(image, aspect="auto", origin="lower", extent=extent, cmap="magma")
    ax.axvspan(signal_window_rel[0], signal_window_rel[1], color="cyan", alpha=0.20)
    ax.axvline(signal_peak_rel, color="white", ls="--", alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel("time in chunk [s]")
    ax.set_ylabel("frequency [Hz]")
fig.colorbar(im, ax=axes, label=r"$\log_{10}$ normalized WDM power")
fig.suptitle("WDM normalized time-frequency maps")
fig.tight_layout()
PLOT_WDM_MAP = PLOT_DIR / f"10_coronagraphic_tdi_wdm_tf_maps_{LINK_SOURCE}.png"
fig.savefig(PLOT_WDM_MAP, dpi=150)
plt.show()
print(f"saved {PLOT_WDM_MAP}")


# Same WDM maps without signal-window or peak overlays, to inspect the TF morphology blind.
fig, axes = plt.subplots(1, len(plot_skies), figsize=(7 * len(plot_skies), 4), sharey=True)
if len(plot_skies) == 1:
    axes = [axes]
for ax, name in zip(axes, plot_skies):
    prod = wdm_products[name]
    freqs = prod["freqs_hz"]
    times_rel_wdm = prod["times_rel"]
    fmask = (freqs >= 0.0) & (freqs <= WDM_HIGH_FREQ_CUT)
    image = np.log10(prod["normalized_power"][fmask] + 1e-30)
    extent = [
        times_rel_wdm[0] - 0.5 * prod["tf_template"].dt,
        times_rel_wdm[-1] + 0.5 * prod["tf_template"].dt,
        freqs[fmask][0],
        freqs[fmask][-1],
    ]
    im = ax.imshow(image, aspect="auto", origin="lower", extent=extent, cmap="magma")
    ax.set_title(name)
    ax.set_xlabel("time in chunk [s]")
    ax.set_ylabel("frequency [Hz]")
fig.colorbar(im, ax=axes, label=r"$\log_{10}$ normalized WDM power")
fig.suptitle("WDM normalized time-frequency maps, no signal highlight")
fig.tight_layout()
PLOT_WDM_MAP_NO_HIGHLIGHT = PLOT_DIR / f"10_coronagraphic_tdi_wdm_tf_maps_no_highlight_{LINK_SOURCE}.png"
fig.savefig(PLOT_WDM_MAP_NO_HIGHLIGHT, dpi=150)
plt.show()
print(f"saved {PLOT_WDM_MAP_NO_HIGHLIGHT}")


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3500100306.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/10_coronagraphic_tdi_wdm_tf_stat_noisy.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3500100306.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3500100306.py:64: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3500100306.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3500100306.py:93: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/10_coronagraphic_tdi_wdm_tf_3bins_noisy.png
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/10_coronagraphic_tdi_wdm_tf_maps_noisy.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/10_coronagraphic_tdi_wdm_tf_maps_no_highlight_noisy.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/3500100306.py:96: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save Complex Time-Domain Outputs

In [12]:
out_df = pd.DataFrame({"t_lisa": t_lisa, "t_rel": t_rel})
out_df["is_signal_window"] = (t_lisa >= signal_window_lisa[0]) & (t_lisa <= signal_window_lisa[1])
for name, out in results.items():
    kappa = out["time"]["kappa"]
    out_df[f"kappa_{name}_real"] = np.real(kappa)
    out_df[f"kappa_{name}_imag"] = np.imag(kappa)
    out_df[f"kappa_{name}_abs"] = np.abs(kappa)

OUT_CSV = OUTPUT_DIR / f"coronagraphic_tdi_time_domain_{LINK_SOURCE}.csv"
out_df.to_csv(OUT_CSV, index=False)
print(f"saved {OUT_CSV}")
out_df.head()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/coronagraphic_tdi_time_domain_noisy.csv


,t_lisa,t_rel,is_signal_window,kappa_signal_direction_real,kappa_signal_direction_imag,kappa_signal_direction_abs,kappa_sky_offset_1_real,kappa_sky_offset_1_imag,kappa_sky_offset_1_abs,kappa_sky_offset_2_real,kappa_sky_offset_2_imag,kappa_sky_offset_2_abs,kappa_sky_offset_3_real,kappa_sky_offset_3_imag,kappa_sky_offset_3_abs,kappa_sky_offset_4_real,kappa_sky_offset_4_imag,kappa_sky_offset_4_abs
0,6.140930e+07,0.0,False,40.268597,-1.998455e-13,40.268597,3.810550,-1.333920e-15,3.810550,54.815357,-5.820742e-15,54.815357,-49.201053,-1.474588e-13,49.201053,-26.968142,-5.044643e-14,26.968142
1,6.140930e+07,0.1,False,65.855470,-2.684978e-13,65.855470,1.722191,-3.558686e-15,1.722191,28.153526,-2.984136e-13,28.153526,-32.314906,-4.175826e-14,32.314906,-6.399140,-9.084567e-14,6.399140
2,6.140930e+07,0.2,False,32.821650,-6.772492e-14,32.821650,0.514582,-1.016506e-14,0.514582,-36.698580,-3.109913e-14,36.698580,23.767731,8.609283e-14,23.767731,12.376740,1.722037e-14,12.376740
3,6.140930e+07,0.3,False,-28.589833,-2.881650e-14,28.589833,2.726196,-2.248253e-14,2.726196,-57.808858,1.675533e-13,57.808858,77.696165,-5.670790e-14,77.696165,10.400337,8.231477e-14,10.400337
4,6.140930e+07,0.4,False,-60.905875,1.155139e-13,60.905875,5.726419,-2.015126e-14,5.726419,0.478069,3.187209e-13,0.478069,77.220974,-6.087391e-15,77.220974,-1.354574,9.773735e-14,1.354574


## Plots

In [13]:
plt.figure(figsize=(13, 5))
for name, out in results.items():
    plt.plot(t_rel, np.abs(out["time"]["kappa"]), lw=0.9, label=f"|kappa| {name}")
plt.axvspan(signal_window_rel[0], signal_window_rel[1], color="gold", alpha=0.20, label="GW signal interval")
plt.axvline(signal_peak_rel, color="k", ls="--", alpha=0.6, label="GW signal peak")
plt.xlabel("time in chunk [s]")
plt.ylabel("|kappa(t)| [Hz, complex magnitude]")
plt.title(f"Time-domain coronagraphic TDI from {LINK_SOURCE} six links")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
PLOT_TIME = PLOT_DIR / f"07_coronagraphic_tdi_time_{LINK_SOURCE}.png"
plt.savefig(PLOT_TIME, dpi=150)
plt.show()
print(f"saved {PLOT_TIME}")

plt.figure(figsize=(13, 5))
for name, out in results.items():
    freqs = out["freqs_hz"]
    kf = out["frequency"]["kappa"]
    pos = freqs > 0
    plt.loglog(freqs[pos], np.abs(kf[pos]), lw=0.8, label=f"|kappa_tilde| {name}")
plt.axvspan(1e-5, 1e-2, color="gold", alpha=0.12, label="previous TDI band")
plt.xlabel("frequency [Hz]")
plt.ylabel("|kappa_tilde(f)|")
plt.title(f"Frequency-domain coronagraphic TDI from {LINK_SOURCE} six links")
plt.grid(alpha=0.3, which="both")
plt.legend()
plt.tight_layout()
PLOT_FREQ = PLOT_DIR / f"07_coronagraphic_tdi_frequency_{LINK_SOURCE}.png"
plt.savefig(PLOT_FREQ, dpi=150)
plt.show()
print(f"saved {PLOT_FREQ}")


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/791955456.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/07_coronagraphic_tdi_time_noisy.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/07_coronagraphic_tdi_frequency_noisy.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_18784/791955456.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Notes For The Next Iteration

This notebook intentionally starts with the static-geometry implementation from `coronagraphic_tdi.py`. After testing, the next refinement should decide whether to:

- compute `kappa` on overlapping time windows with geometry refreshed at each window center,
- replace the static delay phases with delay operators from the actual orbit,
- scan a grid of `(beta, lambda)` values,
- and wrap the validated notebook logic into a reusable script module.